In [1]:
from Trips_Extractor import extract_trips_from_pdf
from Lines_Extractor import parse_line_report_pdf

In [2]:
trips_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"
lines_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Lines test.pdf"

trips = extract_trips_from_pdf(trips_pdf)
lines = parse_line_report_pdf(lines_pdf, first_calendar_page=1)

bid_period_start = lines['bid_period_date_range'].get('start')
bid_period_end = lines['bid_period_date_range'].get('end')

In [3]:
bid_period_start = lines['bid_period_date_range'].get('start')
bid_period_end = lines['bid_period_date_range'].get('end')

In [4]:
from pprint import pprint
pprint(lines)

{'bid_period_date_range': {'end': '2023-07-16', 'start': '2023-05-21'},
 'lines': [{'line_number': 9,
            'pay_periods': [{'BT': '42:39',
                             'CT': '72:12',
                             'DD': '12',
                             'DO': '11',
                             'assignments': [{'date': '2023-05-25',
                                              'start_time': '06:37',
                                              'type': 'trip',
                                              'value': 53},
                                             {'date': '2023-05-30',
                                              'start_time': '06:35',
                                              'type': 'trip',
                                              'value': 33},
                                             {'date': '2023-05-31',
                                              'start_time': '06:05',
                                              'type': 'trip',
           

In [5]:
pprint(trips[607])

{'blocks': [{'end': '19:14',
             'flights': [{'arrival': 'CGN',
                          'departure': 'SDF',
                          'end': '16:44',
                          'flight': 'DH UPS 213',
                          'route_flags': ['C'],
                          'route_raw': 'SDF-CGN(C)',
                          'start': '07:17'},
                         {'arrival': 'FRA',
                          'departure': 'CGN',
                          'end': '19:14',
                          'flight': 'GT N/A BUS G',
                          'route_flags': [],
                          'route_raw': 'CGN-FRA',
                          'start': '17:14'}],
             'rest': '12h01',
             'start': '07:17'},
            {'end': '14:55',
             'flights': [{'arrival': 'MUC',
                          'departure': 'FRA',
                          'end': '09:10',
                          'flight': 'DH LH 100',
                          'route_flags': [],
 

In [8]:
from datetime import datetime, timedelta
import re


TRIP_TYPES = {"trip", "trips"}


def parse_duration(value):
    """
    Converts:
        '20h55' -> timedelta(hours=20, minutes=55)
        '66h51' -> timedelta(hours=66, minutes=51)
        '-'     -> None
    """
    if value is None or value == "-":
        return None

    match = re.match(r"^(\d+)h(\d{2})$", str(value).strip())

    if not match:
        return None

    hours = int(match.group(1))
    minutes = int(match.group(2))

    return timedelta(hours=hours, minutes=minutes)


def datetime_at_or_after(reference_dt, time_value):
    if reference_dt is None:
        raise ValueError("reference_dt is None")

    if time_value is None:
        raise ValueError("time_value is None")

    hour, minute = map(int, time_value.split(":"))

    candidate = reference_dt.replace(
        hour=hour,
        minute=minute,
        second=0,
        microsecond=0
    )

    while candidate < reference_dt:
        candidate += timedelta(days=1)

    return candidate

def unique_preserve_order(items):
    result = []

    for item in items:
        if item not in result:
            result.append(item)

    return result


def extract_route_flags(flight):
    flags = []

    if flight.get("route_flags"):
        flags.extend(flight["route_flags"])

    if flight.get("route_flag"):
        flags.append(flight["route_flag"])

    flight_text = str(flight.get("flight", "")).upper().strip()
    route_raw = str(flight.get("route_raw", "")).upper().strip()

    # Deadhead detection
    # Examples:
    # 'DH AA194'    -> 'DH AA'
    # 'DH UPS396'   -> 'DH UPS'
    # 'DH LH1844'   -> 'DH LH'
    # 'DH WN1782-2' -> 'DH WN'
    dh_match = re.match(r"^DH\s+([A-Z]+)", flight_text)

    if dh_match:
        carrier = dh_match.group(1)
        flags.append(f"DH {carrier}")

        # Bus detection
        if "BUS" in flight_text or "BUS" in route_raw:
            flags.append("BUS")

        return unique_preserve_order(flags)


def build_master_assignment(assignment, trips):
    assignment_type = assignment.get("type")

    # -------------------------
    # Non-trip assignment
    # -------------------------
    if assignment_type not in TRIP_TYPES:
        code = assignment.get("value")

        if code is None:
            code = assignment_type

        return {
            "date": assignment.get("date"),
            "code": code
        }

    # -------------------------
    # Trip assignment
    # -------------------------
    trip_id = assignment.get("value")
    trip = trips.get(trip_id)

    if trip is None:
        return {
            "trip_id": trip_id,
            "date": assignment.get("date"),
            "error": f"Trip {trip_id} not found in trips dictionary",
            "flights": []
        }

    assignment_date = assignment.get("date")
    assignment_start_time = assignment.get("start_time")

    if assignment_date is None or assignment_start_time is None:
        return {
            "trip_id": trip_id,
            "premium": trip.get("premium"),
            "tafb": trip.get("tafb"),
            "total_days_gone": None,
            "error": "Missing assignment date or start_time",
            "flights": []
        }

    current_dt = datetime.strptime(
        f"{assignment_date} {assignment_start_time}",
        "%Y-%m-%d %H:%M"
    )

    flight_records = []

    trip_start_dt = None
    trip_end_dt = None

    blocks = trip.get("blocks", [])

    for block_index, block in enumerate(blocks, start=1):
        block_start_time = block.get("start")

        if current_dt is None:
            raise ValueError(
                f"current_dt is None before block {block_index} "
                f"of trip {trip_id}. Previous block probably had missing end time."
            )

        if block_index == 1:
            block_start_dt = current_dt
        else:
            block_start_dt = datetime_at_or_after(current_dt, block_start_time)

        last_flight_end_dt = block_start_dt
        flights = block.get("flights", [])

        for flight_index, flight in enumerate(flights):
            is_first_flight_in_block = flight_index == 0
            is_last_flight_in_block = flight_index == len(flights) - 1

            if is_first_flight_in_block:
                flight_start_dt = datetime_at_or_after(
                    block_start_dt,
                    flight.get("start")
                )
            else:
                flight_start_dt = datetime_at_or_after(
                    last_flight_end_dt,
                    flight.get("start")
                )

            flight_end_dt = datetime_at_or_after(
                flight_start_dt,
                flight.get("end")
            )

            if trip_start_dt is None:
                trip_start_dt = flight_start_dt

            trip_end_dt = flight_end_dt

            record = {
                "start_date": flight_start_dt.date().isoformat(),
                "departure": flight.get("departure"),

                "end_date": flight_end_dt.date().isoformat(),
                "arrival": flight.get("arrival"),

                "route_flags": extract_route_flags(flight),

                "rest": block.get("rest") if is_last_flight_in_block and block.get("rest") != "-" else None
            }

            flight_records.append(record)
            last_flight_end_dt = flight_end_dt

        block_end_time = block.get("end")

        if block_end_time is None:
            raise ValueError(
                f"Missing block end time in trip {trip_id}, block {block_index}"
            )

        block_end_dt = datetime_at_or_after(
            last_flight_end_dt,
            block_end_time
        )

        rest_td = parse_duration(block.get("rest"))

        if rest_td is not None:
            current_dt = block_end_dt + rest_td
        else:
            current_dt = block_end_dt

    if trip_start_dt is not None and trip_end_dt is not None:
        total_days_gone = (
            trip_end_dt.date() - trip_start_dt.date()
        ).days + 1
    else:
        total_days_gone = None

    return {
        "trip_id": trip_id,
        "premium": trip.get("premium"),
        "tafb": trip.get("tafb"),
        "total_days_gone": total_days_gone,
        "flights": flight_records
    }

In [11]:
def hhmm_to_minutes(value):
    """
    Converts:
        '42:39' -> 2559 minutes
        None    -> 0
    """
    if value is None:
        return 0

    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)


def minutes_to_hhmm(total_minutes):
    """
    Converts:
        2559 -> '42:39'
    """
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours}:{minutes:02d}"

def creating_master_line(trips, lines):
    master_lines = {}
    for line in lines['lines']:
        
        total_BT_minutes = 0
        total_CT_minutes = 0
        total_DD = 0
        total_DO = 0
        total_premium = 0.0

        line_num = line['line_number']
        master_lines[line_num] = {'tot_BT':None, 'tot_CT':None, 'tot_DD':None, 'tot_DO':None, 'tot_Premium':None, 'PPs':[]}

        for pp in line["pay_periods"]:
            total_BT_minutes += hhmm_to_minutes(pp.get("BT"))
            total_CT_minutes += hhmm_to_minutes(pp.get("CT"))
            total_DD += int(pp.get("DD"))
            total_DO += int(pp.get("DO") if pp.get("DO") is not None else 0)

            master_pp = {
                "pp": pp.get("pp"),
                "BT": pp.get("BT"),
                "CT": pp.get("CT"),
                "DD": pp.get("DD"),
                "DO": pp.get("DO") if pp.get("DO") is not None else 0,
                "assignments": []
            }

            for assignment in pp["assignments"]:
                master_assignment = build_master_assignment(assignment, trips)
                master_pp["assignments"].append(master_assignment)
                
                if "premium" in master_assignment:
                    total_premium += float(master_assignment.get("premium"))

            master_lines[line_num]["PPs"].append(master_pp)

        master_lines[line_num]["tot_BT"] = minutes_to_hhmm(total_BT_minutes)
        master_lines[line_num]["tot_BT_mins"] = total_BT_minutes
        master_lines[line_num]["tot_CT"] = minutes_to_hhmm(total_CT_minutes)
        master_lines[line_num]["tot_CT_mins"] = total_CT_minutes
        master_lines[line_num]["tot_DD"] = total_DD
        master_lines[line_num]["tot_DO"] = total_DO
        master_lines[line_num]["tot_Premium"] = total_premium
    
    return master_lines

master_lines = creating_master_line(trips, lines)
pprint(master_lines)

{9: {'PPs': [{'BT': '42:39',
              'CT': '72:12',
              'DD': '12',
              'DO': '11',
              'assignments': [{'flights': [{'arrival': 'DFW',
                                            'departure': 'SDF',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'},
                                           {'arrival': 'SDF',
                                            'departure': 'DFW',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'}],
                               'premium': 0.0,
                               'tafb': '9h04',
           

In [16]:
from datetime import date


def add_blockiness_scores(master_lines, bid_start, bid_end):
    """
    Adds blockiness information to each line in master_lines.

    Uses the current master_lines format:

        master_lines[line_number]["PPs"]
        master_lines[line_number]["PPs"][i]["assignments"]
        assignment["flights"]
        assignment["total_days_gone"]
        line["tot_DD"]
        line["tot_DO"]

    Formula:

        blockiness_score =
            0.5 * (sum(days_gone^2) / tot_DD)
            +
            0.5 * (sum(days_between_trips^2) / tot_DO)
    """

    if isinstance(bid_start, str):
        bid_start = date.fromisoformat(bid_start)

    if isinstance(bid_end, str):
        bid_end = date.fromisoformat(bid_end)

    for line_number, line in master_lines.items():

        line_trip_dates = []
        days_gone_list = []

        # ------------------------------------------------------------
        # 1. Collect all trip dates and days_gone from this line
        # ------------------------------------------------------------
        for pp in line["PPs"]:
            for assignment in pp["assignments"]:
                if 'flights' not in assignment:
                    continue

                flights = assignment["flights"]

                start_dates = []
                end_dates = []

                for flight in flights:
                    start_dates.append(date.fromisoformat(flight["start_date"]))
                    end_dates.append(date.fromisoformat(flight["end_date"]))

                trip_start = min(start_dates)
                trip_end = max(end_dates)

                days_gone = assignment["total_days_gone"]

                line_trip_dates.append({
                    "trip_id": assignment["trip_id"],
                    "trip_start": trip_start,
                    "trip_end": trip_end,
                    "days_gone": days_gone,
                })

                days_gone_list.append(days_gone)

        # Sort the trips by date
        line_trip_dates.sort(key=lambda trip: trip["trip_start"])

        # ------------------------------------------------------------
        # 2. Calculate days_between_trips_list
        # ------------------------------------------------------------
        days_between_trips_list = []

        if line_trip_dates:

            # Bid period start to first trip
            first_trip_start = line_trip_dates[0]["trip_start"]
            first_gap = (first_trip_start - bid_start).days
            days_between_trips_list.append(max(first_gap, 0))

            # Between trips
            for i in range(1, len(line_trip_dates)):
                previous_trip_end = line_trip_dates[i - 1]["trip_end"]
                next_trip_start = line_trip_dates[i]["trip_start"]

                gap = (next_trip_start - previous_trip_end).days
                days_between_trips_list.append(max(gap, 0))

            # Last trip to bid period end
            last_trip_end = line_trip_dates[-1]["trip_end"]
            last_gap = (bid_end - last_trip_end).days
            days_between_trips_list.append(max(last_gap, 0))

        else:
            # No trips at all in this line
            full_bid_period_gap = (bid_end - bid_start).days + 1
            days_between_trips_list.append(full_bid_period_gap)

        # ------------------------------------------------------------
        # 3. Calculate blockiness score
        # ------------------------------------------------------------
        tot_DD = line["tot_DD"]
        tot_DO = line["tot_DO"]

        if tot_DD > 0:
            days_gone_component = sum(day ** 2 for day in days_gone_list) / tot_DD
        else:
            days_gone_component = 0

        if tot_DO > 0:
            days_between_component = (
                sum(day ** 2 for day in days_between_trips_list) / tot_DO
            )
        else:
            days_between_component = 0

        blockiness_score = (
            0.5 * days_gone_component
            + 0.5 * days_between_component
        )

        # ------------------------------------------------------------
        # 4. Save the results back into the line
        # ------------------------------------------------------------
        line["blockiness_score"] = blockiness_score


add_blockiness_scores(master_lines, bid_period_start, bid_period_end)

pprint(master_lines)

{9: {'PPs': [{'BT': '42:39',
              'CT': '72:12',
              'DD': '12',
              'DO': '11',
              'assignments': [{'flights': [{'arrival': 'DFW',
                                            'departure': 'SDF',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'},
                                           {'arrival': 'SDF',
                                            'departure': 'DFW',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'}],
                               'premium': 0.0,
                               'tafb': '9h04',
           

In [14]:
from datetime import date, timedelta


def add_blockiness_scores(master_lines, bid_start, bid_end, pp_length_days=28):
    """
    Adds only one top-level key to each line:

        line["blockiness_score"]

    Logic:
        - Each pay period is scored separately.
        - If a pay period contains VTO, VOR, RA, RB, SA, or SB,
          that pay period gets a blockiness score of 0.
        - Normal flying pay periods get a real blockiness score.
        - Final line score is the average of the pay period scores.

    Assumes each PP is 28 days unless pp_length_days is changed.
    """

    if isinstance(bid_start, str):
        bid_start = date.fromisoformat(bid_start)

    if isinstance(bid_end, str):
        bid_end = date.fromisoformat(bid_end)

    zero_score_codes = {"VTO", "VOR", "RA", "RB", "SA", "SB"}

    for line_number, line in master_lines.items():

        pp_scores = []

        for pp_index, pp in enumerate(line["PPs"]):

            pp_work_blocks = []
            pp_has_zero_score_code = False

            # --------------------------------------------------------
            # 1. Determine this PP's real calendar boundary
            # --------------------------------------------------------
            pp_start = bid_start + timedelta(days=pp_index * pp_length_days)
            pp_end = pp_start + timedelta(days=pp_length_days - 1)

            if pp_end > bid_end:
                pp_end = bid_end

            # --------------------------------------------------------
            # 2. Collect normal trip blocks and detect special codes
            # --------------------------------------------------------
            for assignment in pp["assignments"]:

                # Normal trip assignment
                if "flights" in assignment:

                    start_dates = []
                    end_dates = []

                    for flight in assignment["flights"]:
                        start_dates.append(date.fromisoformat(flight["start_date"]))
                        end_dates.append(date.fromisoformat(flight["end_date"]))

                    trip_start = min(start_dates)
                    trip_end = max(end_dates)

                    pp_work_blocks.append({
                        "start_date": trip_start,
                        "end_date": trip_end,
                        "days_gone": assignment["total_days_gone"],
                    })

                # Code assignment: VTO, VOR, RA, RB, SA, SB, etc.
                elif assignment.get("type") == "code":

                    code = assignment["value"]

                    if code in zero_score_codes:
                        pp_has_zero_score_code = True

            # --------------------------------------------------------
            # 3. If this PP contains VTO/VOR/RA/RB/SA/SB, score is 0
            # --------------------------------------------------------
            if pp_has_zero_score_code:
                pp_scores.append(0)
                continue

            # --------------------------------------------------------
            # 4. Calculate normal PP blockiness
            # --------------------------------------------------------
            pp_work_blocks.sort(key=lambda block: block["start_date"])

            days_gone_list = [
                block["days_gone"]
                for block in pp_work_blocks
            ]

            days_between_trips_list = []

            if pp_work_blocks:

                # PP start to first trip
                first_gap = (pp_work_blocks[0]["start_date"] - pp_start).days
                days_between_trips_list.append(max(first_gap, 0))

                # Between trips
                for i in range(1, len(pp_work_blocks)):
                    previous_trip_end = pp_work_blocks[i - 1]["end_date"]
                    next_trip_start = pp_work_blocks[i]["start_date"]

                    gap = (next_trip_start - previous_trip_end).days
                    days_between_trips_list.append(max(gap, 0))

                # Last trip to PP end
                last_gap = (pp_end - pp_work_blocks[-1]["end_date"]).days
                days_between_trips_list.append(max(last_gap, 0))

            else:
                # No trips and no special code.
                # Rare edge case, but keep the score numeric.
                days_between_trips_list.append((pp_end - pp_start).days + 1)

            dd_denominator = int(pp["DD"])
            do_denominator = int(pp["DO"])

            if dd_denominator > 0:
                days_gone_component = (
                    sum(day ** 2 for day in days_gone_list) / dd_denominator
                )
            else:
                days_gone_component = 0

            if do_denominator > 0:
                days_between_component = (
                    sum(day ** 2 for day in days_between_trips_list) / do_denominator
                )
            else:
                days_between_component = 0

            pp_blockiness_score = (
                0.5 * days_gone_component
                + 0.5 * days_between_component
            )

            pp_scores.append(pp_blockiness_score)

        # ------------------------------------------------------------
        # 5. Final line score
        # ------------------------------------------------------------
        if pp_scores:
            line["blockiness_score"] = sum(pp_scores) / len(pp_scores)
        else:
            line["blockiness_score"] = 0

add_blockiness_scores(master_lines, bid_period_start, bid_period_end)

In [15]:
pprint(master_lines)

{9: {'PPs': [{'BT': '42:39',
              'CT': '72:12',
              'DD': '12',
              'DO': '11',
              'assignments': [{'flights': [{'arrival': 'DFW',
                                            'departure': 'SDF',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'},
                                           {'arrival': 'SDF',
                                            'departure': 'DFW',
                                            'end_date': '2023-05-25',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-25'}],
                               'premium': 0.0,
                               'tafb': '9h04',
           